# ЗАГРУЗКА И ПЕРВИЧНЫЙ АНАЛИЗ ДАННЫХ

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error
import time
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor

In [2]:
# Загружаем данные
train = pd.read_csv('C:\\Users\\User\\Desktop\\pyt\\2 Semester\\Educational practice\\ChemAI-Predict-The-Cure\\date\\train.csv')
test = pd.read_csv('C:\\Users\\User\\Desktop\\pyt\\2 Semester\\Educational practice\\ChemAI-Predict-The-Cure\\date\\test.csv')
sample_sub = pd.read_csv('C:\\Users\\User\\Desktop\\pyt\\2 Semester\\Educational practice\\ChemAI-Predict-The-Cure\\date\\sample_submission.csv')

print(f"\nРазмер обучающей выборки: {train.shape}")
print(f"Размер тестовой выборки: {test.shape}")
print(f"Размер файла сабмишна: {sample_sub.shape}")


Размер обучающей выборки: (751, 214)
Размер тестовой выборки: (250, 211)
Размер файла сабмишна: (250, 4)


In [3]:
# Первые строки данных
print("\nПервые 5 строк обучающих данных:")
print(train.head())


Первые 5 строк обучающих данных:
   index    IC50, mM    CC50, mM          SI  MaxAbsEStateIndex  \
0      0  102.414420   95.757483    0.935000           5.466584   
1      1    0.044333    8.401080  189.500000          11.492712   
2      2    4.437964   50.085589   11.285714           5.366084   
3      3    6.827881  682.788051  100.000000          13.317130   
4      4    2.003253   70.001455   34.943894           6.320833   

   MaxEStateIndex  MinAbsEStateIndex  MinEStateIndex       qed        SPS  \
0        5.466584           0.719259        0.719259  0.681165  18.307692   
1       11.492712           0.012350       -3.798024  0.769122  27.652174   
2        5.366084           0.522930        0.522930  0.612606  24.608696   
3       13.317130           0.020658       -4.829339  0.345823  12.400000   
4        6.320833           0.300347        0.300347  0.562066  60.272727   

   ...  fr_sulfide  fr_sulfonamd  fr_sulfone  fr_term_acetylene  fr_tetrazole  \
0  ...           1 

In [4]:
# Информация о типах данных и пропусках
print("\nИнформация о типах данных в train:")
print(train.info())


Информация о типах данных в train:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 751 entries, 0 to 750
Columns: 214 entries, index to fr_urea
dtypes: float64(107), int64(107)
memory usage: 1.2 MB
None


In [5]:
# Проверка на пропуски
print("\nКоличество пропусков в train:")
print(train.isnull().sum().sum())

print("\nКоличество пропусков в test:")
print(test.isnull().sum().sum())


Количество пропусков в train:
24

Количество пропусков в test:
12


In [6]:
# Основная статистика по числовым признакам
print("\nОсновная статистика train:")
print(train.describe())


Основная статистика train:
            index     IC50, mM     CC50, mM            SI  MaxAbsEStateIndex  \
count  751.000000   751.000000   751.000000    751.000000         751.000000   
mean   375.000000   204.544021   577.426098     89.153313          10.860070   
std    216.939316   370.367937   641.515163    788.882198           3.347314   
min      0.000000     0.003517     0.700808      0.011489           2.321942   
25%    187.500000    13.222351    99.998894      1.500000           8.921032   
50%    375.000000    44.069306   376.580899      4.000000          12.197500   
75%    562.500000   206.787402   877.508784     17.372463          13.214245   
max    750.000000  4095.188563  4538.976189  15620.600000          15.933463   

       MaxEStateIndex  MinAbsEStateIndex  MinEStateIndex         qed  \
count      751.000000         751.000000      751.000000  751.000000   
mean        10.860070           0.180064       -0.971890    0.577938   
std          3.347314           0.1

# АНАЛИЗ ПРОПУСКОВ

In [7]:
# Проверяем, в каких колонках есть пропуски
print("Колонки с пропусками в train:")
null_cols_train = train.columns[train.isnull().any()].tolist()
print(f"Найдено {len(null_cols_train)} колонок с пропусками")
print(null_cols_train)

print("\nКолонки с пропусками в test:")
null_cols_test = test.columns[test.isnull().any()].tolist()
print(f"Найдено {len(null_cols_test)} колонок с пропусками")
print(null_cols_test)

Колонки с пропусками в train:
Найдено 12 колонок с пропусками
['MaxPartialCharge', 'MinPartialCharge', 'MaxAbsPartialCharge', 'MinAbsPartialCharge', 'BCUT2D_MWHI', 'BCUT2D_MWLOW', 'BCUT2D_CHGHI', 'BCUT2D_CHGLO', 'BCUT2D_LOGPHI', 'BCUT2D_LOGPLOW', 'BCUT2D_MRHI', 'BCUT2D_MRLOW']

Колонки с пропусками в test:
Найдено 12 колонок с пропусками
['MaxPartialCharge', 'MinPartialCharge', 'MaxAbsPartialCharge', 'MinAbsPartialCharge', 'BCUT2D_MWHI', 'BCUT2D_MWLOW', 'BCUT2D_CHGHI', 'BCUT2D_CHGLO', 'BCUT2D_LOGPHI', 'BCUT2D_LOGPLOW', 'BCUT2D_MRHI', 'BCUT2D_MRLOW']


In [8]:
# Смотрим сколько пропусков в каждой колонке
print("\nКоличество пропусков по колонкам в train:")
null_counts_train = train[null_cols_train].isnull().sum().sort_values(ascending=False)
print(null_counts_train)

print("\nКоличество пропусков по колонкам в test:")
null_counts_test = test[null_cols_test].isnull().sum().sort_values(ascending=False)
print(null_counts_test)


Количество пропусков по колонкам в train:
MaxPartialCharge       2
MinPartialCharge       2
MaxAbsPartialCharge    2
MinAbsPartialCharge    2
BCUT2D_MWHI            2
BCUT2D_MWLOW           2
BCUT2D_CHGHI           2
BCUT2D_CHGLO           2
BCUT2D_LOGPHI          2
BCUT2D_LOGPLOW         2
BCUT2D_MRHI            2
BCUT2D_MRLOW           2
dtype: int64

Количество пропусков по колонкам в test:
MaxPartialCharge       1
MinPartialCharge       1
MaxAbsPartialCharge    1
MinAbsPartialCharge    1
BCUT2D_MWHI            1
BCUT2D_MWLOW           1
BCUT2D_CHGHI           1
BCUT2D_CHGLO           1
BCUT2D_LOGPHI          1
BCUT2D_LOGPLOW         1
BCUT2D_MRHI            1
BCUT2D_MRLOW           1
dtype: int64


In [9]:
# Определяем целевые переменные и признаки
target_cols = ['IC50, mM', 'CC50, mM', 'SI']
feature_cols = [col for col in train.columns if col not in target_cols + ['index']]

print(f"\nКоличество признаков: {len(feature_cols)}")
print(f"Первые 10 признаков: {feature_cols[:10]}")


Количество признаков: 210
Первые 10 признаков: ['MaxAbsEStateIndex', 'MaxEStateIndex', 'MinAbsEStateIndex', 'MinEStateIndex', 'qed', 'SPS', 'MolWt', 'HeavyAtomMolWt', 'ExactMolWt', 'NumValenceElectrons']


# ПРЕДОБРАБОТКА ДАННЫХ

In [10]:
# Создаем логарифмические целевые переменные
train['log_IC50'] = np.log1p(train['IC50, mM'])
train['log_CC50'] = np.log1p(train['CC50, mM'])

print("Целевые переменные после логарифмирования:")
print(train[['IC50, mM', 'log_IC50', 'CC50, mM', 'log_CC50']].head())

Целевые переменные после логарифмирования:
     IC50, mM  log_IC50    CC50, mM  log_CC50
0  102.414420  4.638744   95.757483  4.572208
1    0.044333  0.043378    8.401080  2.240825
2    4.437964  1.693405   50.085589  3.933502
3    6.827881  2.057692  682.788051  6.527648
4    2.003253  1.099696   70.001455  4.262700


In [11]:
# Определяем признаки и цели
feature_cols = [col for col in train.columns if col not in ['index', 'IC50, mM', 'CC50, mM', 'SI', 'log_IC50', 'log_CC50']]
X = train[feature_cols]
y_ic50 = train['log_IC50']
y_cc50 = train['log_CC50']

print(f"\nПризнаки: {X.shape}")
print(f"Цель IC50: {y_ic50.shape}")
print(f"Цель CC50: {y_cc50.shape}")


Признаки: (751, 210)
Цель IC50: (751,)
Цель CC50: (751,)


In [12]:
# Разделяем на обучение и валидацию (80% / 20%)
X_train, X_val, y_train_ic50, y_val_ic50, y_train_cc50, y_val_cc50 = train_test_split(
    X, y_ic50, y_cc50, test_size=0.2, random_state=42
)

print(f"\nОбучающая выборка: {X_train.shape}")
print(f"Валидационная выборка: {X_val.shape}")


Обучающая выборка: (600, 210)
Валидационная выборка: (151, 210)


In [13]:
# Обработка пропусков (заполняем медианой)
imputer = SimpleImputer(strategy='median')

X_train_imp = imputer.fit_transform(X_train)
X_val_imp = imputer.transform(X_val)
X_test_imp = imputer.transform(test[feature_cols])

print(f"\nПропуски обработаны. Новые формы:")
print(f"X_train: {X_train_imp.shape}")
print(f"X_val: {X_val_imp.shape}")
print(f"X_test: {X_test_imp.shape}")


Пропуски обработаны. Новые формы:
X_train: (600, 210)
X_val: (151, 210)
X_test: (250, 210)


In [14]:
# Нормализация (StandardScaler)
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train_imp)
X_val_scaled = scaler.transform(X_val_imp)
X_test_scaled = scaler.transform(X_test_imp)

In [15]:
# Проверяем, что пропусков больше нет
print(f"\nПроверка: есть ли NaN в X_train_scaled? {np.isnan(X_train_scaled).any()}")
print(f"Проверка: есть ли NaN в X_test_scaled? {np.isnan(X_test_scaled).any()}")


Проверка: есть ли NaN в X_train_scaled? False
Проверка: есть ли NaN в X_test_scaled? False


# ОБУЧЕНИЕ И СРАВНЕНИЕ МОДЕЛЕЙ

In [16]:
# Реальные значения для валидации
true_ic50_val = np.expm1(y_val_ic50)
true_cc50_val = np.expm1(y_val_cc50)

In [17]:
# Словарь для хранения результатов
results = {}

In [18]:
# RandomForest
start = time.time()
rf = RandomForestRegressor(n_estimators=300, max_depth=20, min_samples_split=5, 
                           min_samples_leaf=2, random_state=42, n_jobs=-1)
rf.fit(X_train_scaled, y_train_ic50)
rf_cc50 = RandomForestRegressor(n_estimators=300, max_depth=20, min_samples_split=5,
                                min_samples_leaf=2, random_state=42, n_jobs=-1)
rf_cc50.fit(X_train_scaled, y_train_cc50)

pred_rf_ic50 = np.expm1(rf.predict(X_val_scaled))
pred_rf_cc50 = np.expm1(rf_cc50.predict(X_val_scaled))

rmse_rf_ic50 = np.sqrt(mean_squared_error(true_ic50_val, pred_rf_ic50))
rmse_rf_cc50 = np.sqrt(mean_squared_error(true_cc50_val, pred_rf_cc50))
results['RandomForest'] = (rmse_rf_ic50, rmse_rf_cc50)
print(f"   IC50: {rmse_rf_ic50:.2f}, CC50: {rmse_rf_cc50:.2f}, время: {time.time()-start:.1f}с")

   IC50: 427.96, CC50: 496.71, время: 1.3с


In [19]:
# GradientBoosting
start = time.time()
gb = GradientBoostingRegressor(n_estimators=300, max_depth=5, learning_rate=0.05,
                               random_state=42)
gb.fit(X_train_scaled, y_train_ic50)
gb_cc50 = GradientBoostingRegressor(n_estimators=300, max_depth=5, learning_rate=0.05,
                                    random_state=42)
gb_cc50.fit(X_train_scaled, y_train_cc50)

pred_gb_ic50 = np.expm1(gb.predict(X_val_scaled))
pred_gb_cc50 = np.expm1(gb_cc50.predict(X_val_scaled))

rmse_gb_ic50 = np.sqrt(mean_squared_error(true_ic50_val, pred_gb_ic50))
rmse_gb_cc50 = np.sqrt(mean_squared_error(true_cc50_val, pred_gb_cc50))
results['GradientBoosting'] = (rmse_gb_ic50, rmse_gb_cc50)
print(f"   IC50: {rmse_gb_ic50:.2f}, CC50: {rmse_gb_cc50:.2f}, время: {time.time()-start:.1f}с")

   IC50: 419.22, CC50: 513.10, время: 8.9с


In [20]:
# KNN
start = time.time()
knn = KNeighborsRegressor(n_neighbors=5, n_jobs=-1)
knn.fit(X_train_scaled, y_train_ic50)
knn_cc50 = KNeighborsRegressor(n_neighbors=5, n_jobs=-1)
knn_cc50.fit(X_train_scaled, y_train_cc50)

pred_knn_ic50 = np.expm1(knn.predict(X_val_scaled))
pred_knn_cc50 = np.expm1(knn_cc50.predict(X_val_scaled))

rmse_knn_ic50 = np.sqrt(mean_squared_error(true_ic50_val, pred_knn_ic50))
rmse_knn_cc50 = np.sqrt(mean_squared_error(true_cc50_val, pred_knn_cc50))
results['KNN'] = (rmse_knn_ic50, rmse_knn_cc50)
print(f"   IC50: {rmse_knn_ic50:.2f}, CC50: {rmse_knn_cc50:.2f}, время: {time.time()-start:.1f}с")

   IC50: 416.84, CC50: 529.40, время: 1.5с


In [21]:
# SVR
start = time.time()
svr = SVR(kernel='rbf', C=1.0, epsilon=0.1)
svr.fit(X_train_scaled, y_train_ic50)
svr_cc50 = SVR(kernel='rbf', C=1.0, epsilon=0.1)
svr_cc50.fit(X_train_scaled, y_train_cc50)

pred_svr_ic50 = np.expm1(svr.predict(X_val_scaled))
pred_svr_cc50 = np.expm1(svr_cc50.predict(X_val_scaled))

rmse_svr_ic50 = np.sqrt(mean_squared_error(true_ic50_val, pred_svr_ic50))
rmse_svr_cc50 = np.sqrt(mean_squared_error(true_cc50_val, pred_svr_cc50))
results['SVR'] = (rmse_svr_ic50, rmse_svr_cc50)
print(f"   IC50: {rmse_svr_ic50:.2f}, CC50: {rmse_svr_cc50:.2f}, время: {time.time()-start:.1f}с")

   IC50: 446.26, CC50: 500.02, время: 0.1с


In [22]:
# 5. Ансамбль (усреднение 3 лучших моделей)
ensemble_ic50 = (pred_rf_ic50 + pred_gb_ic50 + pred_knn_ic50) / 3
ensemble_cc50 = (pred_rf_cc50 + pred_gb_cc50 + pred_knn_cc50) / 3

rmse_ens_ic50 = np.sqrt(mean_squared_error(true_ic50_val, ensemble_ic50))
rmse_ens_cc50 = np.sqrt(mean_squared_error(true_cc50_val, ensemble_cc50))
results['Ensemble (RF+GB+KNN)'] = (rmse_ens_ic50, rmse_ens_cc50)
print(f"   IC50: {rmse_ens_ic50:.2f}, CC50: {rmse_ens_cc50:.2f}")


   IC50: 416.25, CC50: 491.96


In [23]:
# Вывод сводной таблицы
print("\n" + "="*50)
print("СВОДНАЯ ТАБЛИЦА РЕЗУЛЬТАТОВ")
print("="*50)
print(f"{'Модель':<20} {'IC50 RMSE':<12} {'CC50 RMSE':<12}")
print("-"*50)

for name, (ic50, cc50) in sorted(results.items(), key=lambda x: x[1][0] + x[1][1]):
    print(f"{name:<20} {ic50:<12.2f} {cc50:<12.2f}")

print("="*50)


СВОДНАЯ ТАБЛИЦА РЕЗУЛЬТАТОВ
Модель               IC50 RMSE    CC50 RMSE   
--------------------------------------------------
Ensemble (RF+GB+KNN) 416.25       491.96      
RandomForest         427.96       496.71      
GradientBoosting     419.22       513.10      
KNN                  416.84       529.40      
SVR                  446.26       500.02      


In [24]:
# Находим лучшие модели
best_ic50_model = min(results.items(), key=lambda x: x[1][0])
best_cc50_model = min(results.items(), key=lambda x: x[1][1])

print(f"\nЛучшая модель для IC50: {best_ic50_model[0]} (RMSE: {best_ic50_model[1][0]:.2f})")
print(f"Лучшая модель для CC50: {best_cc50_model[0]} (RMSE: {best_cc50_model[1][1]:.2f})")


Лучшая модель для IC50: Ensemble (RF+GB+KNN) (RMSE: 416.25)
Лучшая модель для CC50: Ensemble (RF+GB+KNN) (RMSE: 491.96)


# АНСАМБЛЬ НА ВСЕХ ДАННЫХ

In [25]:
# Обучаем модели на всех данных
rf_full = RandomForestRegressor(n_estimators=300, max_depth=20, min_samples_split=5,
                                min_samples_leaf=2, random_state=42, n_jobs=-1)
rf_full.fit(X_train_scaled, y_train_ic50)
rf_cc50_full = RandomForestRegressor(n_estimators=300, max_depth=20, min_samples_split=5,
                                     min_samples_leaf=2, random_state=42, n_jobs=-1)
rf_cc50_full.fit(X_train_scaled, y_train_cc50)

gb_full = GradientBoostingRegressor(n_estimators=300, max_depth=5, learning_rate=0.05,
                                    random_state=42)
gb_full.fit(X_train_scaled, y_train_ic50)
gb_cc50_full = GradientBoostingRegressor(n_estimators=300, max_depth=5, learning_rate=0.05,
                                         random_state=42)
gb_cc50_full.fit(X_train_scaled, y_train_cc50)

knn_full = KNeighborsRegressor(n_neighbors=5, n_jobs=-1)
knn_full.fit(X_train_scaled, y_train_ic50)
knn_cc50_full = KNeighborsRegressor(n_neighbors=5, n_jobs=-1)
knn_cc50_full.fit(X_train_scaled, y_train_cc50)

,n_neighbors,5
,weights,'uniform'
,algorithm,'auto'
,leaf_size,30
,p,2
,metric,'minkowski'
,metric_params,None
,n_jobs,-1


In [26]:
# Предсказание для теста
test_pred_rf_ic50 = np.expm1(rf_full.predict(X_test_scaled))
test_pred_gb_ic50 = np.expm1(gb_full.predict(X_test_scaled))
test_pred_knn_ic50 = np.expm1(knn_full.predict(X_test_scaled))

test_pred_rf_cc50 = np.expm1(rf_cc50_full.predict(X_test_scaled))
test_pred_gb_cc50 = np.expm1(gb_cc50_full.predict(X_test_scaled))
test_pred_knn_cc50 = np.expm1(knn_cc50_full.predict(X_test_scaled))


In [27]:
# Ансамбль (среднее арифметическое)
test_ensemble_ic50 = (test_pred_rf_ic50 + test_pred_gb_ic50 + test_pred_knn_ic50) / 3
test_ensemble_cc50 = (test_pred_rf_cc50 + test_pred_gb_cc50 + test_pred_knn_cc50) / 3

In [28]:
# Вычисляем SI
epsilon = 1e-10
test_ensemble_si = test_ensemble_cc50 / (test_ensemble_ic50 + epsilon)
test_ensemble_si = np.clip(test_ensemble_si, 0, 10000)

In [29]:
# Создаем файл для сабмишна
submission = sample_sub.copy()
submission['IC50'] = test_ensemble_ic50
submission['CC50'] = test_ensemble_cc50
submission['SI'] = test_ensemble_si

In [31]:
# Сохраняем
submission.to_csv('C:\\Users\\User\\Desktop\\pyt\\2 Semester\\Educational practice\\ChemAI-Predict-The-Cure\\submissions\\submission.csv', index=False)

print("\nПервые 5 строк:")
print(submission.head())


Первые 5 строк:
   index        IC50        CC50        SI
0      0   65.652191  183.069971  2.788482
1      1  115.204532  326.535088  2.834394
2      2   39.692650  254.537835  6.412720
3      3  151.866274  440.085022  2.897846
4      4   57.768015  180.183666  3.119090


In [32]:
# Небольшая статистика по предсказаниям
print("\nСтатистика предсказаний:")
print(f"IC50: min={test_ensemble_ic50.min():.2f}, max={test_ensemble_ic50.max():.2f}, mean={test_ensemble_ic50.mean():.2f}")
print(f"CC50: min={test_ensemble_cc50.min():.2f}, max={test_ensemble_cc50.max():.2f}, mean={test_ensemble_cc50.mean():.2f}")
print(f"SI: min={test_ensemble_si.min():.2f}, max={test_ensemble_si.max():.2f}, mean={test_ensemble_si.mean():.2f}")


Статистика предсказаний:
IC50: min=0.31, max=1012.58, mean=109.30
CC50: min=10.57, max=3149.04, mean=463.88
SI: min=1.07, max=260.29, mean=13.16


# LIGHTGBM

In [33]:
# Обучаем LightGBM для IC50
start = time.time()

lgb_ic50 = LGBMRegressor(
    n_estimators=500,
    max_depth=7,
    learning_rate=0.03,
    num_leaves=31,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    verbose=-1,
    n_jobs=-1
)
lgb_ic50.fit(X_train_scaled, y_train_ic50)

,boosting_type,'gbdt'
,num_leaves,31
,max_depth,7
,learning_rate,0.03
,n_estimators,500
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


In [34]:
# Обучаем LightGBM для CC50
start = time.time()

lgb_cc50 = LGBMRegressor(
    n_estimators=500,
    max_depth=7,
    learning_rate=0.03,
    num_leaves=31,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    verbose=-1,
    n_jobs=-1
)
lgb_cc50.fit(X_train_scaled, y_train_cc50)

,boosting_type,'gbdt'
,num_leaves,31
,max_depth,7
,learning_rate,0.03
,n_estimators,500
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


In [35]:
# Предсказания на валидации
pred_lgb_ic50 = np.expm1(lgb_ic50.predict(X_val_scaled))
pred_lgb_cc50 = np.expm1(lgb_cc50.predict(X_val_scaled))

In [36]:
# Оценка качества
rmse_lgb_ic50 = np.sqrt(mean_squared_error(true_ic50_val, pred_lgb_ic50))
rmse_lgb_cc50 = np.sqrt(mean_squared_error(true_cc50_val, pred_lgb_cc50))

print(f"\n=== РЕЗУЛЬТАТЫ LIGHTGBM ===")
print(f"IC50 RMSE: {rmse_lgb_ic50:.2f}")
print(f"CC50 RMSE: {rmse_lgb_cc50:.2f}")


=== РЕЗУЛЬТАТЫ LIGHTGBM ===
IC50 RMSE: 416.32
CC50 RMSE: 473.39


In [37]:
# Добавляем LightGBM в ансамбль
# Получаем предсказания всех моделей на валидации
pred_rf_ic50 = np.expm1(rf.predict(X_val_scaled))
pred_gb_ic50 = np.expm1(gb.predict(X_val_scaled))
pred_knn_ic50 = np.expm1(knn.predict(X_val_scaled))

pred_rf_cc50 = np.expm1(rf_cc50.predict(X_val_scaled))
pred_gb_cc50 = np.expm1(gb_cc50.predict(X_val_scaled))
pred_knn_cc50 = np.expm1(knn_cc50.predict(X_val_scaled))

# Ансамбль из 4 моделей (простое среднее)
ensemble4_ic50 = (pred_rf_ic50 + pred_gb_ic50 + pred_knn_ic50 + pred_lgb_ic50) / 4
ensemble4_cc50 = (pred_rf_cc50 + pred_gb_cc50 + pred_knn_cc50 + pred_lgb_cc50) / 4

rmse_ens4_ic50 = np.sqrt(mean_squared_error(true_ic50_val, ensemble4_ic50))
rmse_ens4_cc50 = np.sqrt(mean_squared_error(true_cc50_val, ensemble4_cc50))

print(f"Ансамбль из 4 моделей (RF+GB+KNN+LGB):")
print(f"   IC50: {rmse_ens4_ic50:.2f}")
print(f"   CC50: {rmse_ens4_cc50:.2f}")

Ансамбль из 4 моделей (RF+GB+KNN+LGB):
   IC50: 415.36
   CC50: 483.43


In [38]:
# Сравнительная таблица
print("\n" + "="*60)
print("СРАВНИТЕЛЬНАЯ ТАБЛИЦА ВСЕХ МОДЕЛЕЙ")
print("="*60)
print(f"{'Модель':<25} {'IC50 RMSE':<12} {'CC50 RMSE':<12}")
print("-"*60)

results['LightGBM'] = (rmse_lgb_ic50, rmse_lgb_cc50)
results['Ensemble (4 models)'] = (rmse_ens4_ic50, rmse_ens4_cc50)

for name, (ic50, cc50) in sorted(results.items(), key=lambda x: x[1][0] + x[1][1]):
    print(f"{name:<25} {ic50:<12.2f} {cc50:<12.2f}")

print("="*60)


СРАВНИТЕЛЬНАЯ ТАБЛИЦА ВСЕХ МОДЕЛЕЙ
Модель                    IC50 RMSE    CC50 RMSE   
------------------------------------------------------------
LightGBM                  416.32       473.39      
Ensemble (4 models)       415.36       483.43      
Ensemble (RF+GB+KNN)      416.25       491.96      
RandomForest              427.96       496.71      
GradientBoosting          419.22       513.10      
KNN                       416.84       529.40      
SVR                       446.26       500.02      


In [39]:
# Предсказание для теста (ансамбль из 4 моделей)
# Обучаем LightGBM на всех данных
lgb_ic50_full = LGBMRegressor(
    n_estimators=500, max_depth=7, learning_rate=0.03,
    num_leaves=31, subsample=0.8, colsample_bytree=0.8,
    random_state=42, verbose=-1, n_jobs=-1
)
lgb_ic50_full.fit(X_train_scaled, y_train_ic50)

lgb_cc50_full = LGBMRegressor(
    n_estimators=500, max_depth=7, learning_rate=0.03,
    num_leaves=31, subsample=0.8, colsample_bytree=0.8,
    random_state=42, verbose=-1, n_jobs=-1
)
lgb_cc50_full.fit(X_train_scaled, y_train_cc50)

# Предсказания для теста
test_pred_rf_ic50 = np.expm1(rf_full.predict(X_test_scaled))
test_pred_gb_ic50 = np.expm1(gb_full.predict(X_test_scaled))
test_pred_knn_ic50 = np.expm1(knn_full.predict(X_test_scaled))
test_pred_lgb_ic50 = np.expm1(lgb_ic50_full.predict(X_test_scaled))

test_pred_rf_cc50 = np.expm1(rf_cc50_full.predict(X_test_scaled))
test_pred_gb_cc50 = np.expm1(gb_cc50_full.predict(X_test_scaled))
test_pred_knn_cc50 = np.expm1(knn_cc50_full.predict(X_test_scaled))
test_pred_lgb_cc50 = np.expm1(lgb_cc50_full.predict(X_test_scaled))

# Ансамбль из 4 моделей
test_ensemble4_ic50 = (test_pred_rf_ic50 + test_pred_gb_ic50 + test_pred_knn_ic50 + test_pred_lgb_ic50) / 4
test_ensemble4_cc50 = (test_pred_rf_cc50 + test_pred_gb_cc50 + test_pred_knn_cc50 + test_pred_lgb_cc50) / 4

# Вычисляем SI
test_ensemble4_si = test_ensemble4_cc50 / (test_ensemble4_ic50 + 1e-10)
test_ensemble4_si = np.clip(test_ensemble4_si, 0, 10000)

In [42]:
# Сохраняем
submission_lgb = sample_sub.copy()
submission_lgb['IC50'] = test_ensemble4_ic50
submission_lgb['CC50'] = test_ensemble4_cc50
submission_lgb['SI'] = test_ensemble4_si

submission_lgb.to_csv('C:\\Users\\User\\Desktop\\pyt\\2 Semester\\Educational practice\\ChemAI-Predict-The-Cure\\submissions\\lightgbm_ensemble.csv', index=False)

print("\nПервые 5 строк:")
print(submission_lgb.head())

print("\nСтатистика предсказаний LightGBM ансамбля:")
print(f"IC50: min={test_ensemble4_ic50.min():.2f}, max={test_ensemble4_ic50.max():.2f}, mean={test_ensemble4_ic50.mean():.2f}")
print(f"CC50: min={test_ensemble4_cc50.min():.2f}, max={test_ensemble4_cc50.max():.2f}, mean={test_ensemble4_cc50.mean():.2f}")


Первые 5 строк:
   index        IC50        CC50        SI
0      0   65.928403  169.554233  2.571793
1      1  103.320836  329.553719  3.189615
2      2   40.305599  258.800835  6.420965
3      3  144.674826  400.720917  2.769804
4      4   58.372736  185.864909  3.184105

Статистика предсказаний LightGBM ансамбля:
IC50: min=0.31, max=1020.22, mean=112.64
CC50: min=10.46, max=3165.14, mean=474.01


# LIGHTGBM (ОТДЕЛЬНО)

In [43]:
# Обучаем LightGBM на ВСЕХ данных (train + val)
X_full_scaled = np.vstack([X_train_scaled, X_val_scaled])
y_full_ic50 = np.concatenate([y_train_ic50, y_val_ic50])
y_full_cc50 = np.concatenate([y_train_cc50, y_val_cc50])

lgb_best_ic50 = LGBMRegressor(
    n_estimators=500,
    max_depth=7,
    learning_rate=0.03,
    num_leaves=31,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    verbose=-1,
    n_jobs=-1
)
lgb_best_ic50.fit(X_full_scaled, y_full_ic50)

lgb_best_cc50 = LGBMRegressor(
    n_estimators=500,
    max_depth=7,
    learning_rate=0.03,
    num_leaves=31,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    verbose=-1,
    n_jobs=-1
)
lgb_best_cc50.fit(X_full_scaled, y_full_cc50)

# Предсказываем тест
test_lgb_ic50 = np.expm1(lgb_best_ic50.predict(X_test_scaled))
test_lgb_cc50 = np.expm1(lgb_best_cc50.predict(X_test_scaled))
test_lgb_si = test_lgb_cc50 / (test_lgb_ic50 + 1e-10)
test_lgb_si = np.clip(test_lgb_si, 0, 10000)

In [44]:
# Сохраняем
submission_lightgbm = sample_sub.copy()
submission_lightgbm['IC50'] = test_lgb_ic50
submission_lightgbm['CC50'] = test_lgb_cc50
submission_lightgbm['SI'] = test_lgb_si
submission_lightgbm.to_csv('C:\\Users\\User\\Desktop\\pyt\\2 Semester\\Educational practice\\ChemAI-Predict-The-Cure\\submissions\\submission_lightgbm.csv', index=False)

print("\n Статистика предсказаний:")
print(f"   IC50: min={test_lgb_ic50.min():.2f}, max={test_lgb_ic50.max():.2f}, mean={test_lgb_ic50.mean():.2f}")
print(f"   CC50: min={test_lgb_cc50.min():.2f}, max={test_lgb_cc50.max():.2f}, mean={test_lgb_cc50.mean():.2f}")
print(f"   SI:   min={test_lgb_si.min():.2f}, max={test_lgb_si.max():.2f}, mean={test_lgb_si.mean():.2f}")


 Статистика предсказаний:
   IC50: min=0.14, max=2885.20, mean=165.06
   CC50: min=9.38, max=3442.19, mean=517.24
   SI:   min=0.67, max=824.98, mean=16.48


# XGBOOST

In [45]:
# Обучаем XGBoost для IC50
start = time.time()

xgb_ic50 = XGBRegressor(
    n_estimators=500,
    max_depth=7,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    verbosity=0
)
xgb_ic50.fit(X_train_scaled, y_train_ic50)

,objective,'reg:squarederror'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,0.8
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


In [46]:
# Обучаем XGBoost для CC50
start = time.time()

xgb_cc50 = XGBRegressor(
    n_estimators=500,
    max_depth=7,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    verbosity=0
)
xgb_cc50.fit(X_train_scaled, y_train_cc50)

,objective,'reg:squarederror'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,0.8
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


In [47]:
# Предсказания на валидации
pred_xgb_ic50 = np.expm1(xgb_ic50.predict(X_val_scaled))
pred_xgb_cc50 = np.expm1(xgb_cc50.predict(X_val_scaled))

In [48]:
# Оценка качества
rmse_xgb_ic50 = np.sqrt(mean_squared_error(true_ic50_val, pred_xgb_ic50))
rmse_xgb_cc50 = np.sqrt(mean_squared_error(true_cc50_val, pred_xgb_cc50))

print(f"\n=== РЕЗУЛЬТАТЫ XGBOOST ===")
print(f"IC50 RMSE: {rmse_xgb_ic50:.2f}")
print(f"CC50 RMSE: {rmse_xgb_cc50:.2f}")


=== РЕЗУЛЬТАТЫ XGBOOST ===
IC50 RMSE: 434.97
CC50 RMSE: 495.18


In [49]:
# Сравниваем с LightGBM
print("\n=== СРАВНЕНИЕ XGBOOST vs LIGHTGBM ===")
print(f"{'Модель':<15} {'IC50 RMSE':<12} {'CC50 RMSE':<12}")
print("-"*40)
print(f"{'XGBoost':<15} {rmse_xgb_ic50:<12.2f} {rmse_xgb_cc50:<12.2f}")
print(f"{'LightGBM':<15} {rmse_lgb_ic50:<12.2f} {rmse_lgb_cc50:<12.2f}")


=== СРАВНЕНИЕ XGBOOST vs LIGHTGBM ===
Модель          IC50 RMSE    CC50 RMSE   
----------------------------------------
XGBoost         434.97       495.18      
LightGBM        416.32       473.39      


In [50]:
# Обучаем XGBoost на ВСЕХ данных

X_full_scaled = np.vstack([X_train_scaled, X_val_scaled])
y_full_ic50 = np.concatenate([y_train_ic50, y_val_ic50])
y_full_cc50 = np.concatenate([y_train_cc50, y_val_cc50])

xgb_best_ic50 = XGBRegressor(
    n_estimators=500, max_depth=7, learning_rate=0.03,
    subsample=0.8, colsample_bytree=0.8,
    random_state=42, n_jobs=-1, verbosity=0
)
xgb_best_ic50.fit(X_full_scaled, y_full_ic50)

xgb_best_cc50 = XGBRegressor(
    n_estimators=500, max_depth=7, learning_rate=0.03,
    subsample=0.8, colsample_bytree=0.8,
    random_state=42, n_jobs=-1, verbosity=0
)
xgb_best_cc50.fit(X_full_scaled, y_full_cc50)

# Предсказываем тест
test_xgb_ic50 = np.expm1(xgb_best_ic50.predict(X_test_scaled))
test_xgb_cc50 = np.expm1(xgb_best_cc50.predict(X_test_scaled))
test_xgb_si = test_xgb_cc50 / (test_xgb_ic50 + 1e-10)
test_xgb_si = np.clip(test_xgb_si, 0, 10000)

In [51]:
# Сохраняем
submission_xgb = sample_sub.copy()
submission_xgb['IC50'] = test_xgb_ic50
submission_xgb['CC50'] = test_xgb_cc50
submission_xgb['SI'] = test_xgb_si
submission_xgb.to_csv('C:\\Users\\User\\Desktop\\pyt\\2 Semester\\Educational practice\\ChemAI-Predict-The-Cure\\submissions\\submission_xgboost.csv', index=False)

print("\n Статистика предсказаний XGBoost:")
print(f"   IC50: min={test_xgb_ic50.min():.2f}, max={test_xgb_ic50.max():.2f}, mean={test_xgb_ic50.mean():.2f}")
print(f"   CC50: min={test_xgb_cc50.min():.2f}, max={test_xgb_cc50.max():.2f}, mean={test_xgb_cc50.mean():.2f}")


 Статистика предсказаний XGBoost:
   IC50: min=0.19, max=3861.98, mean=157.30
   CC50: min=6.29, max=3816.84, mean=520.06
